# Act 2 - Lab 1 - Create a Genie Agent

This workshop walks you through creating a fully configured **Genie Agent** (formerly Genie Space) over CPG retail analytics gold tables using two methods:

1. **Genie Code** — interactive AI assistant prompts
2. **Genie Agent REST API** — programmatic creation via `POST /api/2.0/genie/spaces` (optional)

A fully configured agent includes:
- Gold tables connected with column descriptions
- Business instructions and metric definitions
- Join relationships, filters, and measures
- Entity matching on categorical columns
- Starter questions and benchmark questions with expected SQL

## Prerequisites

This workshop assumes the following **3 gold tables** already exist in your catalog/schema:

| Table | Description |
| --- | --- |
| `genie_workshop_catalog.<your_schema>.gold_sales_velocity` | Weekly sales rate per product × store — demand sensing |
| `genie_workshop_catalog.<your_schema>.gold_inventory_health` | Days-of-supply and stock status per product × store |
| `genie_workshop_catalog.<your_schema>.gold_supply_chain_coverage` | DC-to-store fill rates by product × retailer × month |

These tables are produced by the **CPG Retail Analytics Pipeline** (a Lakeflow Spark Declarative Pipeline over the `crisp_retail_distribution_data.examples` schema).

> **Action:** Before running Section 2, update `CATALOG`, `SCHEMA`, and `WAREHOUSE_ID` in the configuration cell to match your environment.

---
# Section 1: Create a Genie Agent using Genie Code

Genie Code is the AI coding assistant built into Databricks. You give it **high-level intent** and it figures out the details — exploring your tables, understanding the schema, and making smart decisions about joins, measures, descriptions, and benchmarks.

**How to use:**
1. Open the Databricks Assistant (click the sparkle icon in the sidebar)
2. Give it the prompt below (adapt the table names to your catalog/schema)
3. Genie Code explores your tables, then creates and configures the agent step by step
4. Navigate to the Genie Agent to verify everything looks right

## Prompt to give Genie Code

Copy the prompt below into the Databricks Assistant. Replace `genie_workshop_catalog.labuserxxxxxx_xxxxxxx` with your own `genie_workshop_catalog.labuserxxxxxx_xxxxxxx` (you can find the value in catalog, inside the catalog `genie_workshop_catalog`.)

> **Key insight:** You don't need to spell out every column description, join condition, or SQL query. Genie Code explores the tables itself and figures out appropriate joins, descriptions, measures, and benchmarks — just like an analyst would.

---

````
Create a new Genie Agent called "CPG Command Center_GC" over these 3 gold tables:
- genie_workshop_catalog.labuserxxxxxx_xxxxxxx.gold_sales_velocity
- genie_workshop_catalog.labuserxxxxxx_xxxxxxx.gold_inventory_health
- genie_workshop_catalog.labuserxxxxxx_xxxxxxx.gold_supply_chain_coverage

This is CPG retail data from Crisp covering 2021-2022 across multiple retailers and suppliers.

Configure the agent fully:
- Add business instructions explaining key CPG metrics (sales velocity, days of supply, stock status, supply status, DC-to-demand ratio)
- Set up join relationships between the tables
- Add useful filters and measures for common CPG analytics
- Describe all important columns
- Enable entity matching on categorical columns (retailer, supplier, state, status fields)
- Add 6 starter questions for business users
- Create 15 benchmark questions with expected SQL
````

That's it — Genie Code handles the rest.

### Tips for Genie Code
- **Start simple, iterate later**: The prompt above is intentionally concise. Genie Code explores your tables and makes smart defaults. You can always refine afterward.
- **Be more specific when needed**: If Genie Code's choices don't match your business rules, follow up with corrections like *"Change the days_of_supply threshold for LOW_STOCK from 7 to 14"* or *"Add a benchmark for revenue at risk from out-of-stock products."*
- **Verification**: After creation, ask: *"Read the space config and verify all knowledge snippets, column descriptions, and benchmarks are configured correctly."*
- **Editing**: You can tell Genie Code to update instructions, add new benchmarks, or change column descriptions at any time.

---
# Optional: Create a Genie Agent using the REST API

This section demonstrates creating a Genie Agent **programmatically** using the `POST /api/2.0/genie/spaces` endpoint. The agent is created in a **single API call** with all configuration embedded in the `serialized_space` JSON payload.

> The demo below uses a representative subset of the full configuration. Expand the lists to add more column descriptions, benchmarks, or starter questions as needed.

Reference: [Genie Agents API documentation](https://docs.databricks.com/aws/en/genie-agents/conversation-api/)

In [0]:
# === CONFIGURATION ===
# Update these to match your environment
CATALOG = "genie_workshop_catalog"
SCHEMA = "labuserxxxxxx_xxxxxxx"  # Change to your schema
WAREHOUSE_ID = "<fill_sql_warehouse_id>"  # SQL warehouse ID (required by the API). You can find it in SQL Warehouses. Click on the SQL Warehouse and look for the ID in the Name field

TABLES = [
    f"{CATALOG}.{SCHEMA}.gold_sales_velocity",
    f"{CATALOG}.{SCHEMA}.gold_inventory_health",
    f"{CATALOG}.{SCHEMA}.gold_supply_chain_coverage",
]

SPACE_DESCRIPTION = """CPG retail performance across sales, inventory, and supply chain (Crisp, 2021-2022).

Key definitions:
- Sales velocity = avg units sold per day at product × store level, weekly
- Days of supply = on_hand / avg daily sales
- Stock status: OUT_OF_STOCK | LOW_STOCK (<7 days) | OVERSTOCK (>60 days) | HEALTHY
- Supply status: NO_DC_STOCK | UNDER_SUPPLIED (ratio<0.5) | OVER_SUPPLIED (>3.0) | BALANCED
- DC-to-demand ratio = avg DC on-hand / total store demand per month

Currency: USD. Weeks start Monday."""

# --- Knowledge snippets ---
JOINS = [
    {"title": "Sales velocity to inventory health",
     "base_table": f"{CATALOG}.{SCHEMA}.gold_sales_velocity",
     "joined_table": f"{CATALOG}.{SCHEMA}.gold_inventory_health",
     "condition": "gold_sales_velocity.product_id = gold_inventory_health.product_id AND gold_sales_velocity.store_id = gold_inventory_health.store_id AND gold_sales_velocity.retailer = gold_inventory_health.retailer",
     "relationship_type": "many_to_one",
     "instruction": "Join to identify fast-selling products with low stock."},
    {"title": "Sales velocity to supply chain coverage",
     "base_table": f"{CATALOG}.{SCHEMA}.gold_sales_velocity",
     "joined_table": f"{CATALOG}.{SCHEMA}.gold_supply_chain_coverage",
     "condition": "gold_sales_velocity.product_id = gold_supply_chain_coverage.product_id AND gold_sales_velocity.retailer = gold_supply_chain_coverage.retailer AND DATE_TRUNC('MONTH', gold_sales_velocity.week_start) = gold_supply_chain_coverage.month_key",
     "relationship_type": "many_to_one",
     "instruction": "Join to link store-level demand with DC supply."},
    {"title": "Inventory health to supply chain coverage",
     "base_table": f"{CATALOG}.{SCHEMA}.gold_inventory_health",
     "joined_table": f"{CATALOG}.{SCHEMA}.gold_supply_chain_coverage",
     "condition": "gold_inventory_health.product_id = gold_supply_chain_coverage.product_id AND gold_inventory_health.retailer = gold_supply_chain_coverage.retailer",
     "relationship_type": "many_to_one",
     "instruction": "Join to correlate store stock status with DC supply."},
]

FILTERS = [
    {"title": "Out of stock products", "sql": "gold_inventory_health.stock_status = 'OUT_OF_STOCK'"},
    {"title": "Low stock products", "sql": "gold_inventory_health.stock_status = 'LOW_STOCK'"},
    {"title": "Under-supplied products from DC", "sql": "gold_supply_chain_coverage.supply_status = 'UNDER_SUPPLIED'"},
]

MEASURES = [
    {"title": "Total revenue", "sql": "SUM(gold_sales_velocity.total_revenue)"},
    {"title": "Total units sold", "sql": "SUM(gold_sales_velocity.total_units_sold)"},
    {"title": "Average days of supply", "sql": "AVG(gold_inventory_health.days_of_supply)"},
    {"title": "Count of out-of-stock products", "sql": "COUNT(DISTINCT CASE WHEN gold_inventory_health.stock_status = 'OUT_OF_STOCK' THEN gold_inventory_health.product_id END)"},
]

# --- Column descriptions (representative subset — add more as needed) ---
COLUMN_DESCRIPTIONS = {
    f"{CATALOG}.{SCHEMA}.gold_sales_velocity": {
        "total_revenue": "Total sales revenue in USD for the product at this store during the week",
        "avg_daily_velocity": "Average units sold per selling day (total_units_sold / selling_days)",
    },
    f"{CATALOG}.{SCHEMA}.gold_inventory_health": {
        "days_of_supply": "Estimated days current inventory will last (on_hand_quantity / avg_daily_units)",
        "stock_status": "OUT_OF_STOCK (0 units), LOW_STOCK (<7 days), OVERSTOCK (>60 days), HEALTHY (7-60 days)",
    },
    f"{CATALOG}.{SCHEMA}.gold_supply_chain_coverage": {
        "dc_to_demand_ratio": "Ratio of avg DC inventory to total store demand. Below 0.5 = supply risk",
        "supply_status": "NO_DC_STOCK, UNDER_SUPPLIED (ratio<0.5), OVER_SUPPLIED (>3.0), BALANCED",
    },
}

ENTITY_MATCHING = {
    f"{CATALOG}.{SCHEMA}.gold_sales_velocity": ["retailer", "supplier", "store_state"],
    f"{CATALOG}.{SCHEMA}.gold_inventory_health": ["retailer", "supplier", "store_state", "stock_status"],
    f"{CATALOG}.{SCHEMA}.gold_supply_chain_coverage": ["retailer", "supplier", "supply_status"],
}

# --- Starter questions ---
STARTER_QUESTIONS = [
    "What are the top 10 fastest-selling products by average daily velocity?",
    "How many products are currently out of stock or low stock, broken down by retailer?",
    "Which products are selling fast but have low stock at the store level?",
]

# --- Benchmarks (representative subset — add more as needed) ---
BENCHMARKS = [
    {"question": "What are the top 10 products by total revenue?",
     "sql": f"SELECT product_name, supplier, SUM(total_revenue) AS total_revenue FROM {CATALOG}.{SCHEMA}.gold_sales_velocity GROUP BY product_name, supplier ORDER BY total_revenue DESC LIMIT 10;"},
    {"question": "How many products are in each stock status category?",
     "sql": f"SELECT stock_status, COUNT(DISTINCT product_id) AS product_count FROM {CATALOG}.{SCHEMA}.gold_inventory_health GROUP BY stock_status ORDER BY product_count DESC;"},
    {"question": "Which products are selling fast but have low stock?",
     "sql": f"SELECT sv.product_name, sv.retailer, sv.avg_daily_velocity, ih.days_of_supply, ih.stock_status FROM {CATALOG}.{SCHEMA}.gold_sales_velocity sv JOIN {CATALOG}.{SCHEMA}.gold_inventory_health ih ON sv.product_id = ih.product_id AND sv.store_id = ih.store_id AND sv.retailer = ih.retailer WHERE ih.stock_status IN ('OUT_OF_STOCK', 'LOW_STOCK') ORDER BY sv.avg_daily_velocity DESC LIMIT 20;"},
    {"question": "What is the total revenue at risk from out-of-stock products?",
     "sql": f"SELECT sv.retailer, SUM(sv.total_revenue) AS revenue_at_risk FROM {CATALOG}.{SCHEMA}.gold_sales_velocity sv JOIN {CATALOG}.{SCHEMA}.gold_inventory_health ih ON sv.product_id = ih.product_id AND sv.store_id = ih.store_id AND sv.retailer = ih.retailer WHERE ih.stock_status = 'OUT_OF_STOCK' GROUP BY sv.retailer ORDER BY revenue_at_risk DESC;"},
    {"question": "Show the monthly supply status breakdown by retailer",
     "sql": f"SELECT retailer, month_key, supply_status, COUNT(DISTINCT product_id) AS product_count FROM {CATALOG}.{SCHEMA}.gold_supply_chain_coverage GROUP BY ALL ORDER BY retailer, month_key, supply_status;"},
]

print(f"✅ Configuration loaded: {len(TABLES)} tables, {len(JOINS)} joins, {len(FILTERS)} filters, {len(MEASURES)} measures")
print(f"   {sum(len(v) for v in COLUMN_DESCRIPTIONS.values())} column descriptions, {len(STARTER_QUESTIONS)} starters, {len(BENCHMARKS)} benchmarks")

In [0]:
import json
import uuid
import re
import requests

# Get workspace URL and token from the current Databricks context
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
WORKSPACE_URL = f"https://{ctx.browserHostName().get()}"
TOKEN = ctx.apiToken().get()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

# Get current user for parent_path
USERNAME = ctx.userName().get()
PARENT_PATH = f"/Workspace/Users/{USERNAME}"

def gen_id():
    """Generate a 32-char lowercase hex ID (required by the Genie API v2)."""
    return uuid.uuid4().hex

def backtick_quote(condition):
    """Convert table.column references to `table`.`column` format for join SQL."""
    return re.sub(r'\b(gold_\w+)\.(\w+)', r'`\1`.`\2`', condition)

print(f"Workspace: {WORKSPACE_URL}")
print(f"User:      {USERNAME}")
print(f"Parent:    {PARENT_PATH}")

In [0]:
# ============================================================
# Build the serialized_space dict (Genie API v2 format)
# ============================================================
# API rules:
#   - version: 2
#   - All IDs: 32-char lowercase hex, sorted alphabetically
#   - Arrays sorted by their sort key (id or identifier)
#   - Text fields are arrays of strings: ["content"]
# ============================================================

# --- Sample / starter questions ---
sample_questions = sorted(
    [{"id": gen_id(), "question": [q]} for q in STARTER_QUESTIONS],
    key=lambda x: x["id"],
)

# --- Column configs per table ---
def build_column_configs(table_name):
    descs = COLUMN_DESCRIPTIONS.get(table_name, {})
    entity_cols = ENTITY_MATCHING.get(table_name, [])
    all_cols = sorted(set(list(descs.keys()) + entity_cols))
    configs = []
    for col in all_cols:
        cfg = {"column_name": col}
        if col in descs:
            cfg["description"] = [descs[col]]
        if col in entity_cols:
            cfg["enable_entity_matching"] = True
            cfg["enable_format_assistance"] = True
        configs.append(cfg)
    return configs  # already sorted by column_name

# --- Data sources (tables sorted by identifier) ---
tables = sorted(
    [{"identifier": t, "column_configs": build_column_configs(t)} for t in TABLES],
    key=lambda x: x["identifier"],
)

# --- Text instructions ---
text_instructions = sorted(
    [{"id": gen_id(), "content": [SPACE_DESCRIPTION]}],
    key=lambda x: x["id"],
)

# --- Join specs ---
RT_MAP = {"many_to_one": "MANY_TO_ONE", "one_to_many": "ONE_TO_MANY", "one_to_one": "ONE_TO_ONE"}
join_specs = sorted(
    [
        {
            "id": gen_id(),
            "left": {
                "identifier": j["base_table"],
                "alias": j["base_table"].split(".")[-1],
            },
            "right": {
                "identifier": j["joined_table"],
                "alias": j["joined_table"].split(".")[-1],
            },
            "sql": [
                backtick_quote(j["condition"]),
                f"--rt=FROM_RELATIONSHIP_TYPE_{RT_MAP[j['relationship_type']]}--",
            ],
        }
        for j in JOINS
    ],
    key=lambda x: x["id"],
)

# --- SQL snippet: filters ---
filters = sorted(
    [
        {"id": gen_id(), "sql": [f["sql"]], "display_name": f["title"]}
        for f in FILTERS
    ],
    key=lambda x: x["id"],
)

# --- SQL snippet: measures ---
measures = sorted(
    [
        {
            "id": gen_id(),
            "sql": [m["sql"]],
            "display_name": m["title"],
            "alias": m["title"].lower().replace(" ", "_").replace("-", "_"),
        }
        for m in MEASURES
    ],
    key=lambda x: x["id"],
)

# --- Benchmarks ---
benchmark_questions = sorted(
    [
        {
            "id": gen_id(),
            "question": [b["question"]],
            "answer": [{"format": "SQL", "content": [b["sql"]]}],
        }
        for b in BENCHMARKS
    ],
    key=lambda x: x["id"],
)

# --- Assemble the full serialized_space ---
serialized_space = {
    "version": 2,
    "config": {
        "sample_questions": sample_questions,
    },
    "data_sources": {
        "tables": tables,
    },
    "instructions": {
        "text_instructions": text_instructions,
        "join_specs": join_specs,
        "sql_snippets": {
            "filters": filters,
            "measures": measures,
        },
    },
    "benchmarks": {
        "questions": benchmark_questions,
    },
}

# --- Summary ---
print("✅ serialized_space built successfully:")
print(f"   Version:           {serialized_space['version']}")
print(f"   Sample questions:  {len(sample_questions)}")
print(f"   Tables:            {len(tables)}")
print(f"   Column configs:    {sum(len(t.get('column_configs', [])) for t in tables)}")
print(f"   Text instructions: {len(text_instructions)}")
print(f"   Join specs:        {len(join_specs)}")
print(f"   Filters:           {len(filters)}")
print(f"   Measures:          {len(measures)}")
print(f"   Benchmarks:        {len(benchmark_questions)}")
print(f"   Payload size:      {len(json.dumps(serialized_space)):,} chars")

In [0]:
# Create the Genie Agent with a single POST call
payload = {
    "title": "CPG Command Center_API",
    "description": SPACE_DESCRIPTION,
    "parent_path": PARENT_PATH,
    "serialized_space": json.dumps(serialized_space),
    "warehouse_id": WAREHOUSE_ID,
}

response = requests.post(
    f"{WORKSPACE_URL}/api/2.0/genie/spaces",
    headers=HEADERS,
    json=payload,
)

result = response.json()
if response.status_code == 200:
    space_id = result.get("space_id") or result.get("id")
    print(f"✅ Genie Agent created successfully!")
    print(f"   Name:     CPG Command Center_API")
    print(f"   Space ID: {space_id}")
    print(f"   URL:      {WORKSPACE_URL}/genie/rooms/{space_id}")
else:
    print(f"❌ Error {response.status_code}:")
    print(json.dumps(result, indent=2))